Dataset: https://www.kaggle.com/datasets/shanegerami/ai-vs-human-text

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import os
import zipfile

In [ ]:
!pip install kaggle joblib > /dev/null

In [ ]:
# Data Loading and Preprocessing
data_file = 'AI_Human.csv'
print("\n2. Loading and cleaning data...")
try:
    df = pd.read_csv(data_file, engine='python', quoting=3, on_bad_lines='skip')
except Exception as e:
    print(f"Error reading CSV: {e}")
    # Fallback to default engine if python engine with quoting=3 fails
    print("Attempting to read with default engine...")
    df = pd.read_csv(data_file, on_bad_lines='skip')

df.columns = ['text', 'generated']
df.dropna(subset=['text', 'generated'], inplace=True)

# only the 'text' column is converted to lowercase
df['text'] = df['text'].astype(str).str.lower()

# Convert 'generated' column to numeric, coercing errors, and drop NaNs before filtering
df['generated'] = pd.to_numeric(df['generated'], errors='coerce')
df.dropna(subset=['generated'], inplace=True)

# Filter out rows where 'generated' is not 0 or 1
df = df[df['generated'].isin([0, 1])]


X = df['text']
y = df['generated']

# DIAGNOSTIC STEP: Print unique values and counts in the target variable
print("\nUnique values and counts in target variable 'y' after cleaning:")
print(y.value_counts())

# Train-Test Split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# 3. Feature Engineering (TF-IDF)

# much larger max_features allows the model to capture distinguishing
# words and phrases between human and AI text.
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 3), # Increased to trigrams for better phrase capturing
    max_features=150000
)

X_train_features = tfidf_vectorizer.fit_transform(X_train)
X_test_features = tfidf_vectorizer.transform(X_test)

print(f"Features shape after TF-IDF: {X_train_features.shape}")

# 4. Model Training and Evaluation
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report # Added import
print(" Training Logistic Regression Model...")
# Removed n_jobs=2 to avoid the UserWarning, using default optimized solver settings
model = LogisticRegression(solver='liblinear', random_state=42, max_iter=1000)
model.fit(X_train_features, y_train) 

print(" Evaluating Model Performance...")
y_pred = model.predict(X_test_features)
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"**NEW ACCURACY on Test Set: {accuracy:.4f}**")
print("Classification Report:\n", report)


2. Loading and cleaning data...

Unique values and counts in target variable 'y' after cleaning:
generated
0.0    10845
1.0     4935
Name: count, dtype: int64
Features shape after TF-IDF: (12624, 136191)
 Training Logistic Regression Model...
 Evaluating Model Performance...
**NEW ACCURACY on Test Set: 0.9189**
Classification Report:
               precision    recall  f1-score   support

         0.0       0.90      0.99      0.94      2166
         1.0       0.97      0.76      0.86       990

    accuracy                           0.92      3156
   macro avg       0.94      0.88      0.90      3156
weighted avg       0.92      0.92      0.92      3156



In [17]:
# Serialization and Saving `.pkl` files (Using joblib)
print("\n Saving model and vectorizer files using joblib...")

VECTORIZER_FILE = 'tfidf_vectorizer.pkl'
MODEL_FILE = 'ai_detector_model.pkl'

# Save the vectorizer
joblib.dump(tfidf_vectorizer, VECTORIZER_FILE)
print(f"Successfully saved Vectorizer to {VECTORIZER_FILE}")

# Save the trained model
joblib.dump(model, MODEL_FILE)
print(f"Successfully saved Model to {MODEL_FILE}")


print("TRAINING COMPLETE. Download the two .pkl files for your Streamlit app.")


 Saving model and vectorizer files using joblib...
Successfully saved Vectorizer to tfidf_vectorizer.pkl
Successfully saved Model to ai_detector_model.pkl
TRAINING COMPLETE. Download the two .pkl files for your Streamlit app.
